# Fase 1 · Zero-shot con Gemma y CLINC150

Este cuaderno es un experimento autocontenido: solo importa bibliotecas externas.
Reproduce la fase 1 del proyecto sin importar `gemma_clinc` ni leer sus YAML.
La fase 1 **no entrena**: evalúa el modelo original sobre el `test` oficial.
Ejecuta las celdas en orden. `limit = 25` es una prueba corta; `None` usa todo `test`.


## 1. Parámetros e importaciones

Las constantes corresponden a `configs/phase1_zero_shot.yaml`. Están aquí para
que se pueda experimentar sin navegar por la arquitectura del proyecto.


In [ ]:
import json
import platform
import random
import re
import urllib.request
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
seed = 42
limit = 25
model_id = "google/gemma-4-E2B-it"
max_new_tokens = 16
system_prompt = (
    "You are an intent classification system. Select exactly one label from the "
    "allowed labels. Use out_of_scope only when none of the other labels applies. "
    "Return only the label name, with no explanation or punctuation."
)
random.seed(seed)
if limit is not None and limit < 1:
    raise ValueError("limit debe ser positivo")
print("Proyecto:", root)

## 2. Descargar y leer CLINC150

Respetamos el orden oficial: primero consultas normales, luego las OOS.
Las etiquetas salen de `train`, pero ningún ejemplo de `train` entra en el prompt.


In [ ]:
source_url = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json"
dataset_path = root / "data/raw/clinc150_full.json"
if not dataset_path.exists():
    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(source_url, headers={"User-Agent": "gemma-clinc/0.1"})
    with urllib.request.urlopen(request, timeout=60) as response:
        dataset_path.write_bytes(response.read())
with dataset_path.open(encoding="utf-8") as file:
    dataset = json.load(file)


def examples(split):
    rows = list(dataset[split]) + list(dataset[f"oos_{split}"])
    return [
        {
            "id": f"{split}-{i:05d}",
            "text": text,
            "label": "out_of_scope" if label == "oos" else label,
        }
        for i, (text, label) in enumerate(rows)
    ]


train_rows = examples("train")
test_rows = examples("test")
test_rows = test_rows[:limit] if limit is not None else test_rows
labels = sorted({row["label"] for row in train_rows if row["label"] != "out_of_scope"})
labels.append("out_of_scope")
print(len(test_rows), "consultas de test;", len(labels), "etiquetas")
print(test_rows[0])

## 3. Construir el prompt

La instrucción, las 151 etiquetas permitidas y una consulta son toda la entrada.


In [ ]:
label_block = "\n".join(f"- {label}" for label in labels)


def messages_for(text):
    user_prompt = f"Allowed labels:\n{label_block}\n\nUser query:\n{text}"
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


print(messages_for(test_rows[0]["text"])[1]["content"][:450] + "...")

## 4. Cargar Gemma en 4 bits

El modelo cuantizado se ejecuta en GPU. Su tabla de embeddings por capa ocupa
varios GB: la dejamos en RAM y transferimos únicamente las filas consultadas.
Los módulos de audio y visión no participan en este experimento de texto.


In [ ]:
device_map = {"": 0, "model.language_model.embed_tokens_per_layer": "cpu"}
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    device_map=device_map,
    dtype=torch.float16,
    quantization_config=quantization,
)

embedding = model.model.language_model.embed_tokens_per_layer
hook = getattr(embedding, "_hf_hook", None)
hooks = getattr(hook, "hooks", (hook,))
weights_map = next(
    (
        h.weights_map
        for h in hooks
        if getattr(h, "offload", False) and getattr(h, "weights_map", None) is not None
    ),
    None,
)
if weights_map is None:
    raise RuntimeError("No se encontró la tabla de embeddings descargada en CPU")
cpu_weight = weights_map["weight"].detach()
if cpu_weight.device.type != "cpu":
    raise RuntimeError("La tabla de embeddings no está en CPU")


class CpuPerLayerEmbedding(nn.Module):
    def __init__(self, weight, padding_idx, scale):
        super().__init__()
        self.weight = weight  # Congelado: no entra en el estado del adaptador.
        self.padding_idx = padding_idx
        self.scale = scale

    def forward(self, input_ids):
        rows = F.embedding(input_ids.cpu(), self.weight, self.padding_idx)
        return (rows * self.scale).to(input_ids.device)


model.model.language_model.embed_tokens_per_layer = CpuPerLayerEmbedding(
    cpu_weight,
    embedding.padding_idx,
    embedding.scalar_embed_scale,
)
model.model.vision_tower = None
model.model.audio_tower = None
model.model.embed_vision = None
model.model.embed_audio = None
model.eval()
print("Gemma cargado")

## 5. Interpretar las respuestas y calcular métricas

Una salida inválida queda como `None` y cuenta como error. Mantenemos el mismo
parser y las mismas métricas que el experimento de software.


In [ ]:
def parse_label(raw_output):
    normalized = raw_output.strip().lower().replace("-", "_").replace(" ", "_")
    normalized = normalized.strip("`'\".,:;!()[]{}")
    if normalized in labels:
        return normalized
    wrapped = re.search(r"(?:intent|label)_*[:=]_*([a-z0-9_]+)", normalized)
    if wrapped and wrapped.group(1) in labels:
        return wrapped.group(1)
    found = {
        label
        for label in labels
        if re.search(rf"(?<![a-z0-9_]){re.escape(label)}(?![a-z0-9_])", normalized)
    }
    return next(iter(found)) if len(found) == 1 else None


def measure(y_true, y_pred):
    predictions = [p if p is not None else "__invalid__" for p in y_pred]
    in_scope = [i for i, truth in enumerate(y_true) if truth != "out_of_scope"]
    oos_true = [truth == "out_of_scope" for truth in y_true]
    oos_pred = [pred == "out_of_scope" for pred in predictions]
    precision, recall, oos_f1, _ = precision_recall_fscore_support(
        oos_true,
        oos_pred,
        average="binary",
        zero_division=0,
    )
    confusions = Counter(
        (truth, pred) for truth, pred in zip(y_true, predictions, strict=True) if truth != pred
    )
    return {
        "n_examples": len(y_true),
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(
            y_true,
            predictions,
            labels=sorted(set(y_true)),
            average="macro",
            zero_division=0,
        ),
        "in_scope_accuracy": accuracy_score(
            [y_true[i] for i in in_scope],
            [predictions[i] for i in in_scope],
        ),
        "oos_precision": precision,
        "oos_recall": recall,
        "oos_f1": oos_f1,
        "invalid_output_rate": predictions.count("__invalid__") / len(predictions),
        "top_confusions": [
            {"true": truth, "predicted": pred, "count": count}
            for (truth, pred), count in confusions.most_common(20)
        ],
    }

## 6. Evaluar y guardar

Conservamos el texto bruto generado para auditar las predicciones. Los JSON
resultantes tienen el mismo esquema que los de la aplicación.


In [ ]:
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
run_dir = root / "artifacts/phase1" / f"phase1_zero_shot_{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)
config_snapshot = {
    "name": "phase1_zero_shot",
    "seed": seed,
    "dataset": {
        "source_url": source_url,
        "cache_path": "data/raw/clinc150_full.json",
        "split": "test",
        "include_oos": True,
    },
    "model": {
        "id": model_id,
        "revision": None,
        "quantization": "4bit",
        "dtype": "float16",
        "device_map": {"": 0},
        "offload_per_layer_embeddings": True,
        "text_only": True,
    },
    "prompt": {"system": system_prompt, "include_label_descriptions": False},
    "generation": {"max_new_tokens": max_new_tokens, "do_sample": False},
    "output": {"root_dir": "artifacts/phase1", "checkpoint_every": 25},
    "training": None,
}


def write_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


write_json(
    run_dir / "run_metadata.json",
    {
        "run_id": run_id,
        "started_at": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "limit": limit,
        "config": config_snapshot,
    },
)

y_true, y_pred = [], []
for example in tqdm(test_rows, desc="Zero-shot evaluation", unit="example"):
    inputs = processor.apply_chat_template(
        messages_for(example["text"]),
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)
    prompt_length = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    raw_output = processor.decode(output_ids[0][prompt_length:], skip_special_tokens=True)
    prediction = parse_label(raw_output)
    record = {
        "id": example["id"],
        "text": example["text"],
        "expected": example["label"],
        "predicted": prediction,
        "raw_output": raw_output,
        "correct": prediction == example["label"],
    }
    with (run_dir / "predictions.jsonl").open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    y_true.append(example["label"])
    y_pred.append(prediction)
    if len(y_true) % 25 == 0:
        write_json(run_dir / "metrics.partial.json", measure(y_true, y_pred))

metrics = measure(y_true, y_pred)
write_json(run_dir / "metrics.json", metrics)
(run_dir / "metrics.partial.json").unlink(missing_ok=True)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("Artefactos:", run_dir)